# Training ablation studies

This notebook is intentionally thin: `ablation_studies` rebuilds the canonical 400×400 baseline and changes only the mechanism declared by cases 0–5. Every epoch uses the same deterministic panel and the independent AdaFace, DEX and KID evaluator. Inference ablations 101–104 are intentionally deferred.

In [ ]:
from pathlib import Path

from src.ablation_studies import ablation_studies

DATASET_ROOT = Path('/content/drive/MyDrive/dataset_unificado')
FGNET_ROOT = Path('/content/FGNET/FGNET/images')
MONITOR_IMAGE = DATASET_ROOT / 'id_100' / '26.jpg'
OUTPUT_ROOT = Path('/content/drive/MyDrive/face_aging_ablations')

# These two images must be the same person at two real longitudinal ages.
EVAL_SOURCE_IMAGE = 'https://your-storage/person_x_age_20.jpg'
EVAL_TARGET_IMAGE = 'https://your-storage/person_x_age_40.jpg'

## Independent evaluation checkpoints

All evaluator files are local and are never downloaded by the runner. One source/target pair is enough for AdaFace and DEX, but KID needs `>=2` same-protocol longitudinal pairs and is noisy on a small epoch panel. For paper evaluation, pass `evaluation_pairs=[...]` with multiple same-person pairs; omitting explicit pairs uses the fixed validation split.

In [ ]:
METRICS_CONFIG = {
    'adaface_repo_path': '/content/AdaFace',
    'adaface_checkpoint_path': '/content/checkpoints/adaface_ir101_webface12m.ckpt',
    'dex_prototxt_path': '/content/checkpoints/dex_age.prototxt',
    'dex_checkpoint_path': '/content/checkpoints/dex_age.caffemodel',
    'kid_inception_weights_path': '/content/checkpoints/weights-inception-2015-12-05-6726825d.pth',
    'device': 'cpu',  # keeps evaluation models away from training VRAM
    'local_files_only': True,
    'evaluation': {
        'batch_size': 16,
        'kid_seed': 2026,
    },
}

## Fast wiring smoke test

This runs one optimizer step for case 3 and still exercises the epoch evaluator. Use a fresh output directory or set `overwrite=True`.

In [ ]:
smoke_report = ablation_studies(
    case=3,
    dataset_root=DATASET_ROOT,
    kaggle_path=FGNET_ROOT,
    monitoring_image=MONITOR_IMAGE,
    monitoring_source_age=26,
    evaluation_source_image=EVAL_SOURCE_IMAGE,
    evaluation_target_image=EVAL_TARGET_IMAGE,
    evaluation_source_age=20,
    evaluation_target_age=40,
    evaluation_identity_id='person_x',
    metrics_config=METRICS_CONFIG,
    num_epochs=1,
    max_train_steps=1,
    epoch_eval_size=1,
    final_eval_size=1,
    output_root=OUTPUT_ROOT / 'smoke',
    device='cuda',
    overwrite=True,
)
smoke_report

## Canonical sequential run

Case 0 is the full baseline; 1 removes numerical conditioning; 2 removes DoRA without replacing it; 3 removes relative-age loss; 4 removes zero-delta anchors and preservation loss while retaining small-delta weighting; 5 removes identity loss. `None` overrides inherit the notebook baseline (14 epochs, DoRA LR 2e-5, 39 inference steps).

In [ ]:
reports = ablation_studies(
    case=[0, 1, 2, 3, 4, 5],
    dataset_root=DATASET_ROOT,
    kaggle_path=FGNET_ROOT,
    monitoring_image=MONITOR_IMAGE,
    monitoring_source_age=26,
    evaluation_source_image=EVAL_SOURCE_IMAGE,
    evaluation_target_image=EVAL_TARGET_IMAGE,
    evaluation_source_age=20,
    evaluation_target_age=40,
    evaluation_identity_id='person_x',
    metrics_config=METRICS_CONFIG,
    num_epochs=14,
    lr_lora=2e-5,
    lr_conv_in=3e-6,
    lr_age_conditioner=7.5e-5,
    num_inference_steps=39,
    strength=0.35,
    evaluate_each_epoch=True,
    epoch_eval_size=64,
    final_eval_size=None,
    output_root=OUTPUT_ROOT,
    device='cuda',
)
reports